In [4]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
import operator
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
load_dotenv()

from typing import Literal
from pydantic import Field,BaseModel
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain_core.messages import BaseMessage
from langgraph.graph import add_messages
from langgraph.checkpoint.memory import InMemorySaver
import time

In [3]:
class crashstate(TypedDict):
    input:str
    step1: str
    step2: str
    step3: str

In [5]:
def step1(state:crashstate)->crashstate:
    print("state 1 executed")
    return{'step1':"done",'input':state['input']}
def step2(state:crashstate)->crashstate:
    print("step 2 hanging.... manualy interrupt the kernel to simulate crash")
    time.sleep(30)
    return{'step2':"done"}
def step3(state:crashstate)->crashstate:
    print("state 3 executed")
    return{'done':True}


In [7]:
graph=StateGraph(crashstate)
graph.add_node("step1",step1)
graph.add_node("step2",step2)
graph.add_node("step3",step3)

graph.add_edge(START,"step1")
graph.add_edge("step1","step2")
graph.add_edge("step2","step3")
graph.add_edge("step3",END)

checkpoint_saver=InMemorySaver()
workflow=graph.compile(checkpointer=checkpoint_saver)

In [16]:
config2={"configurable":{"thread_id":"2"}}

In [17]:
try:
    print("workflow started, please interrupt  during step2   kernel to simulate crash")
    workflow.invoke({"input":"start"},config=config2)
except KeyboardInterrupt:
    print("workflow interrupted, simulating crash")

workflow started, please interrupt  during step2   kernel to simulate crash
state 1 executed
step 2 hanging.... manualy interrupt the kernel to simulate crash
workflow interrupted, simulating crash


In [18]:
workflow.get_state(config2)

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1995d6-0ab2-6338-8001-92000ff88ab6'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-16T10:29:35.803261+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1995d6-0ab0-6339-8000-d759a40ff129'}}, tasks=(PregelTask(id='3ff5bc14-e647-06f3-f826-a5e9d981f737', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [19]:
workflow.get_state_history(config2)

<generator object Pregel.get_state_history at 0x7e2903019640>

In [20]:
workflow.invoke(None,config=config2)

step 2 hanging.... manualy interrupt the kernel to simulate crash
state 3 executed


{'input': 'start', 'step1': 'done', 'step2': 'done'}

In [21]:
workflow.get_state(config2)

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1995d9-7b8e-63f9-8003-e9b75859f192'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-16T10:31:08.168076+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1995d9-7b8b-6bc9-8002-1abce8614821'}}, tasks=(), interrupts=())

In [22]:
workflow.get_state_history(config2)

<generator object Pregel.get_state_history at 0x7e2903019900>